# Run MODFLOW-2000 with FloPy

Local FloPy workflow (no Tapis calls):
1. Stage an existing model from shared model storage into a local run workspace
2. Load the staged model for inspection with FloPy
3. Run the original model input files without rewriting legacy packages
4. Check expected output files

This notebook is a compact local FloPy workflow for running an existing MODFLOW-2000 model. It is intentionally shorter than the Gulf model notebook, but the same pattern applies: configure paths, prepare the model workspace, run the model, and verify outputs before interpretation.

## Before You Run

- Make sure the model input files are available from the configured shared storage path.
- Make sure the correct MODFLOW executable is available for this model version.
- Review the path variables before running cells that stage files or write outputs.

## Expected Outputs

- A local tutorial run workspace is created under `model_output_directory/`.
- The existing model is loaded or staged for inspection with FloPy.
- The model run reports whether it succeeded.
- Expected model output files are checked before moving on.


## Imports And Path Setup

This cell imports FloPy and supporting utilities, then sets the local paths used by the run. Review the model source, staged workspace, and executable settings before continuing.


In [ ]:
from pathlib import Path

import flopy

from modflow_utils import stage_model_workspace


## Use Shared Model Inputs

The notebook uses model inputs and MODFLOW executables that are already staged on the shared TACC filesystem, then prepares a local tutorial workspace for generated output.


In [ ]:
# Paths and executable configuration
modeldir = Path(r"/corral-repl/tacc/aci/PT2050/projects/PTDATAX-272/workingGAMs/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000")
run_dir = Path(r"model_output_directory/modflow_2000")
exe_name = r"/corral-repl/tacc/aci/PT2050/community/DSO-Institute-2026/flopy/bin/mf2000"

STAGE_OVERWRITE = True

print(f"FloPy version: {flopy.__version__}")
print(f"Source model directory: {modeldir}")
print(f"Run workspace: {run_dir}")
print(f"Executable: {exe_name}")

if not modeldir.exists():
    raise FileNotFoundError(f"Model directory not found: {modeldir}")

exe_path = Path(exe_name)
if not exe_path.is_file():
    raise FileNotFoundError(f"MODFLOW executable not found: {exe_path}")

print(f"Found executable: {exe_path}")
run_dir.mkdir(parents=True, exist_ok=True)
staged_modeldir = stage_model_workspace(modeldir, run_dir, overwrite=STAGE_OVERWRITE)
print(f"Staged model workspace: {staged_modeldir}")


## Load The Existing Model

FloPy loads the staged MODFLOW-2000 model for inspection. Use this step to confirm that the name file and package files are being read from the intended workspace.


In [ ]:
preferred_namefiles = ["ygjk_tr.nam", "model.nam"]
namefile = next((n for n in preferred_namefiles if (staged_modeldir / n).exists()), None)
if namefile is None:
    nam_candidates = sorted(staged_modeldir.glob("*.nam"))
    if not nam_candidates:
        raise FileNotFoundError(f"No MODFLOW-2000 .nam file found in {staged_modeldir}")
    namefile = nam_candidates[0].name

print(f"Using name file: {namefile}")

model_obj = flopy.modflow.Modflow.load(
    f=namefile,
    version="mf2k",
    exe_name=exe_name,
    model_ws=str(staged_modeldir),
    check=False,
    verbose=True,
)


## Use Staged Original Inputs

This legacy MODFLOW-2000 workflow runs the staged original input files without rewriting packages through FloPy. This avoids changing older package formatting.


In [ ]:
# Legacy MF2K packages are run from the staged original input files.
# Rewriting STR input through FloPy can change formatting and break parsing.
print(f"Using staged input files in {staged_modeldir}; skipping FloPy write_input().")


## Run The Model

This cell launches MODFLOW-2000 through FloPy and reports whether the run succeeded. Stop here if the model does not terminate normally.


In [ ]:
# Run model through FloPy
run_method = getattr(model_obj, "run_simulation", None)
if callable(run_method):
    success, buff = run_method()
else:
    success, buff = model_obj.run_model(silent=False, report=True)

print(f"Success: {success}")
if not success:
    print("Model did not terminate normally.")


## Check Expected Outputs

This cell confirms that expected output files were created in the staged run workspace.


In [ ]:
# Output checks
list_candidates = sorted(run_dir.glob("*.lst"))
for f in list_candidates[:5]:
    print(f"List file: {f.name}")

if not list_candidates:
    print("No .lst file found yet in run workspace.")
